# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

sns.set_style('whitegrid')
pio.templates.default = 'plotly_white'

In [2]:
df = pd.read_csv('../data/vehicles.csv')

In [3]:
df.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
0,7222695916,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az
1,7218891961,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar
2,7221797935,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl
3,7222270760,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma
4,7210384030,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   region        426880 non-null  object 
 2   price         426880 non-null  int64  
 3   year          425675 non-null  float64
 4   manufacturer  409234 non-null  object 
 5   model         421603 non-null  object 
 6   condition     252776 non-null  object 
 7   cylinders     249202 non-null  object 
 8   fuel          423867 non-null  object 
 9   odometer      422480 non-null  float64
 10  title_status  418638 non-null  object 
 11  transmission  424324 non-null  object 
 12  VIN           265838 non-null  object 
 13  drive         296313 non-null  object 
 14  size          120519 non-null  object 
 15  type          334022 non-null  object 
 16  paint_color   296677 non-null  object 
 17  state         426880 non-null  object 
dtypes: f

In [ ]:
df_clean =df.copy()
#df_clean.drop(columns=['title_status', 'VIN', 'drive', 'size', 'type', 'paint_color'], inplace=True)
df_clean.fillna({'condition':'Unknown'}, inplace=True)

In [19]:
df_clean = df.copy()
def fill_missing_by_group(df, target_cols, group_cols):
    for col in target_cols:
        missing_before = df[col].isna().sum()

        # compute mode and disagreement % per group
        def group_stats(x):
            if x.dropna().empty:
                return None, 0, 0
            mode_val = x.mode().iloc[0]
            non_null = x.dropna()
            diff_count = (non_null != mode_val).sum()
            diff_pct = diff_count / len(non_null) * 100
            return mode_val, diff_count, diff_pct

        stats = df.groupby(group_cols)[col].agg(
            mode=lambda x: x.mode().iloc[0] if not x.dropna().empty else None,
            diff_count=lambda x: (x.dropna() != x.mode().iloc[0]).sum() if not x.dropna().empty else 0,
            diff_pct=lambda x: (x.dropna() != x.mode().iloc[0]).sum() / len(x.dropna()) * 100 if not x.dropna().empty else 0
        )

        # show groups where values disagree with the mode
        disagreements = stats[stats['diff_count'] > 0].sort_values('diff_pct', ascending=False)
        if not disagreements.empty:
            print(f"\n{col}: groups with values differing from mode ({len(disagreements)} groups):")
            print(disagreements.head(20).to_string())
        else:
            print(f"\n{col}: all groups are unanimous")

        # fill missing values
        mode_map = stats['mode']
        mask = df[col].isna()
        keys = df.loc[mask, group_cols]
        filled = keys.set_index(group_cols).index.map(mode_map)
        df.loc[mask, col] = filled.values
        missing_after = df[col].isna().sum()
        print(f"{col}: {missing_before} missing -> {missing_after} remaining ({missing_before - missing_after} filled)")
    return df

target_cols = ['cylinders', 'fuel', 'transmission', 'size', 'drive', 'type']
group_cols = ['manufacturer', 'model', 'year']

df_clean = fill_missing_by_group(df_clean, target_cols, group_cols)


cylinders: groups with values differing from mode (4097 groups):
                                          mode  diff_count   diff_pct
manufacturer  model       year                                       
ram           2500        1995.0  10 cylinders           6  66.666667
ford          f150        2020.0   5 cylinders           2  66.666667
mercedes-benz benz ml500  2006.0   4 cylinders           2  66.666667
saturn        l300        2001.0   4 cylinders           2  66.666667
ford          thunderbird 1983.0   4 cylinders           2  66.666667
              ranger      1984.0   4 cylinders           2  66.666667
mercedes-benz benz        1978.0   4 cylinders           2  66.666667
ford          bronco      1988.0   4 cylinders           2  66.666667
chevrolet     venture van 2004.0   3 cylinders           2  66.666667
gmc           envoy xuv   2004.0   4 cylinders           3  60.000000
              canyon      2012.0   4 cylinders           3  60.000000
mitsubishi    mirage g4 

In [20]:
df_clean[(df_clean.manufacturer == 'chevrolet') & (df_clean.model == 'volt') & (df_clean.year == 2014)]

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
28482,7313659013,gold country,9295,2014.0,chevrolet,volt,excellent,4 cylinders,gas,107000.0,clean,automatic,1G1RE6E48EU158957,fwd,compact,hatchback,NaN,ca
35753,7311705493,mendocino county,9295,2014.0,chevrolet,volt,excellent,4 cylinders,gas,107000.0,clean,automatic,1G1RE6E48EU158957,fwd,compact,hatchback,NaN,ca
36072,7313657323,merced,9295,2014.0,chevrolet,volt,excellent,4 cylinders,gas,107000.0,clean,automatic,1G1RE6E48EU158957,fwd,compact,hatchback,NaN,ca
37540,7313658419,modesto,9295,2014.0,chevrolet,volt,excellent,4 cylinders,gas,107000.0,clean,automatic,1G1RE6E48EU158957,fwd,compact,hatchback,NaN,ca
37679,7313052469,modesto,12950,2014.0,chevrolet,volt,excellent,4 cylinders,hybrid,40000.0,clean,automatic,NaN,fwd,compact,hatchback,silver,ca
38193,7311023859,modesto,9295,2014.0,chevrolet,volt,excellent,4 cylinders,gas,107000.0,clean,automatic,1G1RE6E48EU158957,fwd,compact,hatchback,NaN,ca
38963,7307575508,modesto,10250,2014.0,chevrolet,volt,NaN,4 cylinders,hybrid,101324.0,clean,automatic,1G1RE6E4XEU170740,fwd,compact,hatchback,white,ca
63920,7316068921,stockton,9499,2014.0,chevrolet,volt,NaN,4 cylinders,other,83496.0,clean,automatic,1G1RE6E42EU161255,fwd,compact,hatchback,NaN,ca
64334,7314908956,stockton,12950,2014.0,chevrolet,volt,excellent,4 cylinders,hybrid,40000.0,clean,automatic,NaN,fwd,compact,hatchback,silver,ca
65736,7308411468,stockton,12950,2014.0,chevrolet,volt,excellent,4 cylinders,hybrid,40000.0,clean,automatic,NaN,fwd,compact,hatchback,silver,ca


In [24]:
df['VIN'].value_counts()

VIN
1FMJU1JT1HEA52352    261
3C6JR6DT3KG560649    235
1FTER1EH1LLA36301    231
5TFTX4CN3EX042751    227
1GCHTCE37G1186784    214
                    ... 
1GCEK19J78Z219711      1
JA4AT3AW1AZ006543      1
4T1BF28B61U153724      1
JTHCF5C25A5041393      1
SAJGX2749VCOO8376      1
Name: count, Length: 118246, dtype: int64

In [29]:
df_ford_expedtion = df[df.VIN == '1FMJU1JT1HEA52352']

In [8]:
df_clean.drop(columns=['title_status', 'VIN', 'paint_color'], inplace=True, errors='ignore')
df_clean.dropna(inplace=True)


In [15]:
df_test = df.copy()
df_test.drop(columns=['title_status', 'region', 'VIN', 'paint_color', 'cylinders', 'fuel', 'transmission', 'size', 'drive', 'type'], inplace=True, errors='ignore')
df_test.dropna(inplace=True)
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 237800 entries, 27 to 426879
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            237800 non-null  int64  
 1   price         237800 non-null  int64  
 2   year          237800 non-null  float64
 3   manufacturer  237800 non-null  object 
 4   model         237800 non-null  object 
 5   condition     237800 non-null  object 
 6   odometer      237800 non-null  float64
 7   state         237800 non-null  object 
dtypes: float64(2), int64(2), object(4)
memory usage: 16.3+ MB


In [9]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 282097 entries, 28 to 426868
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            282097 non-null  int64  
 1   region        282097 non-null  object 
 2   price         282097 non-null  int64  
 3   year          282097 non-null  float64
 4   manufacturer  282097 non-null  object 
 5   model         282097 non-null  object 
 6   condition     282097 non-null  object 
 7   cylinders     282097 non-null  object 
 8   fuel          282097 non-null  object 
 9   odometer      282097 non-null  float64
 10  transmission  282097 non-null  object 
 11  drive         282097 non-null  object 
 12  size          282097 non-null  object 
 13  type          282097 non-null  object 
 14  state         282097 non-null  object 
dtypes: float64(2), int64(2), object(11)
memory usage: 34.4+ MB


### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`. 

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.